In [ ]:
SAGITTAL_DIR = r""
AXIAL_DIR = r""
CORONAL_DIR = r""

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import templateflow.api as tf
import warnings
import tifffile
import SimpleITK as sitk
from skimage.filters import threshold_li, threshold_otsu
from skimage.morphology import closing, disk
from skimage.measure import label, regionprops
from skimage.transform import resize

warnings.filterwarnings("ignore", category=Image.DecompressionBombWarning)

PROCESS_SIZE = 512
SUPPORTED = {".png", ".jpg", ".jpeg", ".jfif", ".bmp", ".tif", ".tiff", ".czi"}
AXIS_SLICE_SEARCH = {"coronal": (0.05, 0.95, 15), "sagittal": (0.05, 0.95, 15), "axial": (0.05, 0.95, 25)}
AXIS_DIM = {"sagittal": 0, "coronal": 1, "axial": 2}

atlas_path = tf.get("MNI152NLin2009cAsym", resolution=1, suffix="T1w")[0]
_sitk = sitk.ReadImage(str(atlas_path))
_ori = sitk.DICOMOrientImageFilter()
_ori.SetDesiredCoordinateOrientation("RAS")
_sitk = _ori.Execute(_sitk)
_sitk = sitk.PermuteAxes(_sitk, [2, 1, 0])
atlas_vol = sitk.GetArrayFromImage(_sitk).astype(np.float32)

def _resize_to(arr: np.ndarray, size: int) -> np.ndarray:
    h, w = arr.shape
    scale = size / max(h, w)
    new_h, new_w = max(1, int(h * scale)), max(1, int(w * scale))
    return resize(arr, (new_h, new_w), anti_aliasing=True, order=1).astype(np.float32)

def get_atlas_candidates(axis: str) -> list[np.ndarray]:
    start, end, n = AXIS_SLICE_SEARCH[axis]
    dim = AXIS_DIM[axis]
    fracs = np.linspace(start, end, n)
    slices = []
    for frac in fracs:
        idx = int(atlas_vol.shape[dim] * frac)
        sl = atlas_vol[idx] if dim == 0 else (atlas_vol[:, idx] if dim == 1 else atlas_vol[:, :, idx])
        sl = np.rot90(sl, k=-1)
        if axis == "axial":
            sl = np.flipud(sl)
        sl = _resize_to(sl.astype(np.float32), PROCESS_SIZE)
        slices.append(sl)
    return slices

ATLAS_CANDIDATES = {axis: get_atlas_candidates(axis) for axis in AXIS_DIM}
ATLAS_RAW = {axis: ATLAS_CANDIDATES[axis][len(ATLAS_CANDIDATES[axis]) // 2] for axis in AXIS_DIM}

In [ ]:
def normalize(arr: np.ndarray) -> np.ndarray:
    lo, hi = float(arr.min()), float(arr.max())
    return (arr - lo) / (hi - lo + 1e-8)

def load_slice(path) -> np.ndarray:
    path = Path(path)
    ext = path.suffix.lower()
    if ext in (".png", ".jpg", ".jpeg", ".jfif", ".bmp"):
        img = Image.open(path)
        if ext in (".jpg", ".jpeg"):
            img.draft("L", (PROCESS_SIZE * 2, PROCESS_SIZE * 2))
        arr = np.array(img.convert("L"), dtype=np.float32)
    elif ext in (".tif", ".tiff"):
        arr = tifffile.imread(str(path)).astype(np.float32)
        if arr.ndim == 3:
            arr = arr.mean(2) if arr.shape[2] <= 4 else arr[arr.shape[0] // 2]
    elif ext == ".czi":
        import czifile
        arr = czifile.imread(str(path)).squeeze().astype(np.float32)
        if arr.ndim == 3:
            arr = arr.mean(2) if arr.shape[2] <= 4 else arr[arr.shape[0] // 2]
    else:
        raise ValueError(f"Unsupported format: {ext}")
    return _resize_to(arr, PROCESS_SIZE)

def is_valid_slice(path) -> bool:
    try:
        ext = Path(path).suffix.lower()
        if ext in (".png", ".jpg", ".jpeg", ".jfif", ".bmp"):
            with Image.open(path) as img:
                w, h = img.size
                return min(h, w) >= 64
        if ext in (".tif", ".tiff"):
            with tifffile.TiffFile(str(path)) as tif:
                h, w = tif.pages[0].shape[:2]
                return min(h, w) >= 64
        return True
    except Exception:
        return False

def detect_polarity(img: np.ndarray) -> int:
    n = normalize(img)
    h, w = n.shape
    s = max(h // 8, w // 8, 10)
    border = np.concatenate([n[:s, :s].ravel(), n[:s, -s:].ravel(), n[-s:, :s].ravel(), n[-s:, -s:].ravel()])
    return 1 if np.median(n[h // 4:3 * h // 4, w // 4:3 * w // 4]) >= np.median(border) else -1

def _ncc(a: np.ndarray, b: np.ndarray) -> float:
    a = a - a.mean()
    b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a ** 2).sum() * (b ** 2).sum()) + 1e-8))

def _ncc_polarity_aware(inp: np.ndarray, atlas: np.ndarray) -> float:
    inp_r = normalize(resize(inp.astype(np.float32), atlas.shape, anti_aliasing=True, order=1))
    atlas_n = normalize(atlas.astype(np.float32))
    if detect_polarity(inp) != detect_polarity(atlas_n):
        inp_r = 1.0 - inp_r
    return _ncc(inp_r, atlas_n)

def best_rotation(img: np.ndarray, atlas_slice: np.ndarray):
    best_score, best_k = -np.inf, 0
    nccs = []
    for k in range(4):
        candidate = np.rot90(img, k=k)
        ncc_score = _ncc_polarity_aware(candidate, atlas_slice)
        nccs.append(ncc_score)
        if ncc_score > best_score:
            best_score, best_k = ncc_score, k
    sorted_nccs = sorted(nccs, reverse=True)
    margin = float(sorted_nccs[0] - sorted_nccs[1])
    rotated = np.rot90(img, k=best_k)
    return rotated, best_k, float(best_k * 90), float(nccs[best_k]), margin

def resolve_flip(img: np.ndarray, atlas_slice: np.ndarray):
    candidates = {"none": img, "lr": np.fliplr(img), "ud": np.flipud(img), "both": np.flipud(np.fliplr(img))}
    best_ncc, best_key = -np.inf, "none"
    for key, cand in candidates.items():
        score = _ncc_polarity_aware(cand, atlas_slice)
        if score > best_ncc:
            best_ncc, best_key = score, key
    return candidates[best_key], (best_key if best_key != "none" else None)

def find_best_atlas_slice(img: np.ndarray, axis: str) -> np.ndarray:
    candidates = ATLAS_CANDIDATES[axis]
    img_small = _resize_to(img.astype(np.float32), 128)
    fast_scores = []
    for i, sl in enumerate(candidates):
        sl_small = _resize_to(sl.astype(np.float32), 128)
        score = max(_ncc_polarity_aware(np.rot90(img_small, k=k), sl_small) for k in (0, 2))
        fast_scores.append((i, score))
    fast_scores.sort(key=lambda x: -x[1])
    top_indices = [i for i, _ in fast_scores[:3]]
    best_ncc, best_sl = -np.inf, ATLAS_RAW[axis]
    for i in top_indices:
        sl = candidates[i]
        ncc = max(_ncc_polarity_aware(np.rot90(img, k=k), sl) for k in range(4))
        if ncc > best_ncc:
            best_ncc, best_sl = ncc, sl
    return best_sl

def _top_wider_than_bottom(mask: np.ndarray) -> float:
    h = mask.shape[0]
    mid = h // 2
    top_width = float(mask[mid:, :].sum(axis=1).mean())
    bottom_width = float(mask[:mid, :].sum(axis=1).mean())
    return top_width - bottom_width

def _symmetry_score(mask: np.ndarray) -> float:
    flipped = np.fliplr(mask)
    inter = float((mask & flipped).sum())
    union = float((mask | flipped).sum())
    return inter / (union + 1e-8)

def _mask_for_landmark(arr: np.ndarray) -> np.ndarray:
    lo, hi = float(arr.min()), float(arr.max())
    norm = (arr - lo) / (hi - lo + 1e-8)
    black_bg = norm < 0.05
    lbl = label(black_bg)
    props = regionprops(lbl)
    if props:
        largest = max(props, key=lambda p: p.area)
        if largest.area / norm.size > 0.10:
            r0, c0, r1, c1 = largest.bbox
            norm = norm[r0:r1, c0:c1]
    for candidate in (norm, 1.0 - norm):
        try:
            thresh = float(threshold_li(candidate))
        except Exception:
            thresh = float(threshold_otsu(candidate))
        mask = closing(candidate > thresh, disk(5)).astype(bool)
        if mask.sum() / mask.size >= 0.05:
            return mask
    thresh = float(threshold_otsu(norm))
    return (norm > thresh).astype(bool)

## Axial

In [ ]:
def orient_axial_landmark(img: np.ndarray) -> tuple[np.ndarray, int]:
    results = []
    for k in range(4):
        rot = np.rot90(img, k=k)
        mask = _mask_for_landmark(rot)
        top_score = _top_wider_than_bottom(mask)
        sym_score = _symmetry_score(mask)
        results.append((k, top_score, sym_score))
    eligible = [(k, ts, ss) for k, ts, ss in results if ts > 0]
    if eligible:
        best_k = max(eligible, key=lambda x: x[2])[0]
    else:
        top_vals = np.array([r[1] for r in results], dtype=float)
        rng = top_vals.max() - top_vals.min() + 1e-8
        best_k = max(range(4), key=lambda k: (results[k][1] - top_vals.min()) / rng + results[k][2])
    return np.rot90(img, k=best_k), best_k

## Coronal

In [ ]:
def orient_coronal_landmark(img: np.ndarray) -> tuple[np.ndarray, int]:
    results = []
    for k in range(4):
        rot = np.rot90(img, k=k)
        mask = _mask_for_landmark(rot)
        top_score = _top_wider_than_bottom(mask)
        sym_score = _symmetry_score(mask)
        results.append((k, top_score, sym_score))
    eligible = [(k, ts, ss) for k, ts, ss in results if ts > 0]
    if eligible:
        best_k = max(eligible, key=lambda x: x[2])[0]
    else:
        top_vals = np.array([r[1] for r in results], dtype=float)
        rng = top_vals.max() - top_vals.min() + 1e-8
        best_k = max(range(4), key=lambda k: (results[k][1] - top_vals.min()) / rng + results[k][2])
    return np.rot90(img, k=best_k), best_k

def _coronal_cerebellum_ud_check(img: np.ndarray) -> bool | None:
    lo, hi = float(img.min()), float(img.max())
    norm = (img - lo) / (hi - lo + 1e-8)
    mask = _mask_for_landmark(norm)
    mh, mw = mask.shape
    labeled = label(mask)
    props = sorted(regionprops(labeled), key=lambda p: -p.area)
    if len(props) < 2:
        return None
    total_fg = mask.sum()
    min_area = max(int(mask.size * 0.01), 100)
    max_area = int(total_fg * 0.40)
    best_score = 0.0
    best_cy = None
    for p in props[1:]:
        if p.area < min_area or p.area > max_area:
            continue
        cy, cx = p.centroid
        bottom_frac = cy / (mh + 1e-8)
        blob_px = np.argwhere(labeled == p.label)
        in_bot = np.sum(blob_px[:, 0] >= mh // 2)
        area_score = float(in_bot) / (p.area + 1e-8)
        combined = 0.5 * bottom_frac + 0.5 * area_score
        if combined > best_score:
            best_score = combined
            best_cy = cy
    if best_cy is None:
        return None
    return best_cy < mh / 2

## Sagittal

In [ ]:
def _cerebellum_score_bottom_right(img: np.ndarray) -> float:
    lo, hi = float(img.min()), float(img.max())
    norm = (img - lo) / (hi - lo + 1e-8)
    mask = _mask_for_landmark(norm)
    mh, mw = mask.shape
    labeled = label(mask)
    props = sorted(regionprops(labeled), key=lambda p: -p.area)
    if len(props) < 2:
        return 0.0
    total_fg = mask.sum()
    min_area = max(int(mask.size * 0.01), 100)
    max_area = int(total_fg * 0.40)
    best_score = 0.0
    for p in props[1:]:
        if p.area < min_area or p.area > max_area:
            continue
        cy, cx = p.centroid
        right_frac = cx / (mw + 1e-8)
        bottom_frac = cy / (mh + 1e-8)
        position_score = right_frac * bottom_frac
        blob_px = np.argwhere(labeled == p.label)
        in_br = np.sum((blob_px[:, 1] >= mw // 2) & (blob_px[:, 0] >= mh // 2))
        area_score = float(in_br) / (p.area + 1e-8)
        combined = 0.5 * position_score + 0.5 * area_score
        if combined > best_score:
            best_score = combined
    return best_score

def _cerebellum_score_bottom(img: np.ndarray) -> float:
    lo, hi = float(img.min()), float(img.max())
    norm = (img - lo) / (hi - lo + 1e-8)
    mask = _mask_for_landmark(norm)
    mh, mw = mask.shape
    labeled = label(mask)
    props = sorted(regionprops(labeled), key=lambda p: -p.area)
    if len(props) < 2:
        return 0.0
    total_fg = mask.sum()
    min_area = max(int(mask.size * 0.01), 100)
    max_area = int(total_fg * 0.40)
    best_score = 0.0
    for p in props[1:]:
        if p.area < min_area or p.area > max_area:
            continue
        cy, cx = p.centroid
        bottom_frac = cy / (mh + 1e-8)
        blob_px = np.argwhere(labeled == p.label)
        in_bot = np.sum(blob_px[:, 0] >= mh // 2)
        area_score = float(in_bot) / (p.area + 1e-8)
        combined = 0.5 * bottom_frac + 0.5 * area_score
        if combined > best_score:
            best_score = combined
    return best_score

def sagittal_landmark_check(img: np.ndarray) -> tuple[np.ndarray, bool]:
    atlas_slice = find_best_atlas_slice(img, "sagittal")
    transforms = {}
    for k in range(4):
        rot = np.rot90(img, k=k)
        transforms[(k, False)] = rot
        transforms[(k, True)] = np.fliplr(rot)
    ncc_scores = {key: _ncc_polarity_aware(cand, atlas_slice) for key, cand in transforms.items()}
    best_key = max(ncc_scores, key=lambda k: ncc_scores[k])
    stage1_img = transforms[best_key]
    candidate_a = stage1_img
    candidate_b = np.fliplr(stage1_img)
    score_a = _cerebellum_score_bottom_right(candidate_a)
    score_b = _cerebellum_score_bottom_right(candidate_b)
    if score_b > score_a + 0.05:
        corrected = candidate_b
    else:
        corrected = candidate_a
    candidate_ud_a = corrected
    candidate_ud_b = np.flipud(corrected)
    ud_score_a = _cerebellum_score_bottom(candidate_ud_a)
    ud_score_b = _cerebellum_score_bottom(candidate_ud_b)
    if ud_score_b > ud_score_a + 0.05:
        corrected = candidate_ud_b
    was_changed = not np.array_equal(corrected, img)
    return corrected, was_changed

In [ ]:
axis_dirs = {"sagittal": SAGITTAL_DIR, "axial": AXIAL_DIR, "coronal": CORONAL_DIR}
found_axes = {}
for axis, p in axis_dirs.items():
    if not p:
        raise ValueError(f"{axis} directory is empty")
    path = Path(p)
    if not path.exists():
        raise FileNotFoundError(f"{axis} directory not found: {path}")
    found_axes[axis] = path

results: dict[str, list[dict]] = {"sagittal": [], "axial": [], "coronal": []}
for axis, folder in found_axes.items():
    files = [f for f in folder.iterdir() if f.suffix.lower() in SUPPORTED and is_valid_slice(f)]
    for fpath in files:
        img = normalize(load_slice(fpath))
        atlas_slice = find_best_atlas_slice(img, axis)
        if axis == "axial":
            final, best_k = orient_axial_landmark(img)
            angle_deg = float(best_k * 90)
            flip = None
        elif axis == "coronal":
            final, best_k = orient_coronal_landmark(img)
            angle_deg = float(best_k * 90)
            flip = None
            ud_flip = _coronal_cerebellum_ud_check(final)
            if ud_flip is True:
                final = np.flipud(final)
        else:
            rotated, k, angle_deg, _, _ = best_rotation(img, atlas_slice)
            final, flip = resolve_flip(rotated, atlas_slice)
            final, _ = sagittal_landmark_check(final)
        results[axis].append({"filename": fpath.name, "original": img, "reoriented": final, "angle_deg": angle_deg, "flip": flip})
    print(f"{axis}: {len(results[axis])} processed")

## Visualization

In [ ]:
axes_list = list(results.keys())
n_cols = 6
for axis in axes_list:
    axis_samples = results[axis]
    n = len(axis_samples)
    if n == 0:
        continue
    n_rows = int(np.ceil(n / 3))
    fig, grid = plt.subplots(n_rows * 2, n_cols, figsize=(n_cols * 3, n_rows * 2 * 3))
    if n_rows * 2 == 1:
        grid = grid[np.newaxis, :]
    for idx, s in enumerate(axis_samples):
        row_pair = idx // 3
        col_pair = (idx % 3) * 2
        grid[row_pair * 2, col_pair].imshow(s["original"], cmap="gray", origin="lower")
        grid[row_pair * 2, col_pair].set_title(f"IN\n{s['filename'][:15]}", fontsize=6)
        grid[row_pair * 2, col_pair].axis("off")
        grid[row_pair * 2, col_pair + 1].imshow(s["reoriented"], cmap="gray", origin="lower")
        grid[row_pair * 2, col_pair + 1].set_title(f"OUT rot={s['angle_deg']:.0f}° fl={s['flip']}", fontsize=6)
        grid[row_pair * 2, col_pair + 1].axis("off")
    for idx in range(len(axis_samples), n_rows * 3):
        row_pair = idx // 3
        col_pair = (idx % 3) * 2
        grid[row_pair * 2, col_pair].axis("off")
        grid[row_pair * 2, col_pair + 1].axis("off")
    for col in range(n_cols):
        for row in range(1, n_rows * 2, 2):
            grid[row, col].axis("off")
    plt.suptitle(f"{axis} — Input vs Reoriented", fontsize=11)
    plt.tight_layout()
    plt.show()